# Christina Doty

In [1]:
import pandas as pd
import numpy as np
import os
import pathlib as Path

# Tidy Crop Data

In [ ]:
def tidy_crop(read_path, write_path, crop='wheat', granularity='state'):
    if granularity == 'state':
        keep_cols = ["year", "state", "state_abbr", "commodity", "statistic_category", "value"]
        pivot_index_cols = ["year", "state", "state_abbr", "commodity"]
    elif granularity == 'county':
        keep_cols = ["year", "state", "state_abbr", "county", "commodity", "statistic_category", "value"]
        pivot_index_cols = ["year", "state", "state_abbr", "county", "commodity"]
    else:
        print("granularity should be either state or county")
        return None
    
    if crop == 'corn':
        keep_cols.append("util_practice_desc")

    df = pd.read_csv(read_path)
    # remove the duplicated yield rows that have dollar amounts instead of BU
    df_no_money = df.loc[df['unit'] != '$']
    df_usefulcols = df_no_money[keep_cols]

    if crop == 'wheat':
        df_pivot = df_usefulcols.pivot_table(index=pivot_index_cols, columns="statistic_category", values="value", aggfunc="first")
        df_pivot.columns.name=None
        df_tidy = df_pivot.reset_index()
        print(f"Sanity check, this should be 4: {len(df_usefulcols) / len(df_tidy)}")
    elif crop == 'corn':
        # Split into planted and harvest groups
        df_planted = df_usefulcols[df_usefulcols["util_practice_desc"] == "ALL UTILIZATION PRACTICES"]
        df_harvest = df_usefulcols[df_usefulcols["util_practice_desc"].isin(["GRAIN", "SILAGE"])]

        # For harvest rows, combine the stat category and util practice into one label
        # # e.g. "area harvested grain", "production silage", "yield grain" etc.
        df_harvest = df_harvest.copy()
        df_harvest["stat_label"] = (df_harvest["statistic_category"] + " " + df_harvest["util_practice_desc"].str.lower())
        
        # remove the extra area planted entries that appear for the grain category
        df_harvest = df_harvest[df_harvest["statistic_category"] != "AREA PLANTED"]

        # For planted rows, the label is just the stat category
        df_planted = df_planted.copy()
        df_planted["stat_label"] = df_planted["statistic_category"]

        # Combine and pivot once on the new label
        df_combined = pd.concat([df_harvest, df_planted], ignore_index=True)

        df_tidy = df_combined.pivot_table(index=pivot_index_cols, columns="stat_label", values="value", aggfunc="first").reset_index()
        df_tidy.columns.name = None
    else:
        print("crop should be either wheat or corn")
        return None
    
    df_tidy.to_csv(write_path)
    return df_tidy

In [ ]:
wheat_state_tidy = tidy_crop("../data/wheat_state.csv", "../data/wheat_state_tidy.csv", crop='wheat', granularity='state')
print(len(wheat_state_tidy))
wheat_state_tidy.head(5)

In [ ]:
wheat_county_tidy = tidy_crop("../data/wheat_county.csv", "../data/wheat_county_tidy.csv", crop='wheat', granularity='county')
print(len(wheat_county_tidy))
wheat_county_tidy.head(5)

In [ ]:
corn_state_tidy = tidy_crop("../data/corn_state.csv", "../data/corn_state_tidy.csv", crop='corn', granularity='state')
print(len(corn_state_tidy))
corn_state_tidy.head(5)

In [ ]:
corn_county_tidy = tidy_crop("../data/corn_county.csv", "../data/corn_county_tidy.csv", crop='corn', granularity='county')
print(len(corn_county_tidy))
corn_county_tidy.head(5)

# Tidy emissions data

In [36]:
def tidy_emissions(read_path, write_path):
    df = pd.read_csv(read_path)
    df_agr = df[df["sector"] == "agriculture"].copy()
    
    subsector_names = {
        "crop-residues": "crop_residues",
        "enteric-fermentation-cattle-pasture": "livestock",
        "enteric-fermentation-cattle-operation": "livestock",
        "enteric-fermentation-other": "livestock",
        "manure-left-on-pasture-cattle": "livestock",
        "manure-applied-to-soils": "manure_fertilizer",
        "manure-management-cattle-operation": "livestock",
        "manure-management-other": "livestock",
        "synthetic-fertilizer-application": "synth_fertilizer",
        "other-agricultural-soil-emissions": "soil",
        "rice-cultivation": "rice",
        "cropland-fires": "crop_fire"
    }

    df_agr["source"] = [subsector_names[subsector] for subsector in df_agr["subsector"]]
    df_agr = df_agr[df_agr["source"] != "livestock"][["year", "state", "admin", "source", "gas", "emissionsQuantity"]]

    df_agr = df_agr.copy()
    df_agr["emission"] = (df_agr["source"] + "_" + df_agr["gas"].str.lower())

    df_pivot = df_agr.pivot_table(index=["year", "state", "admin"], columns="emission", values="emissionsQuantity", aggfunc="sum").reset_index()
    df_pivot.columns.name = None

    df_pivot.to_csv(write_path)
    return df_pivot

read_path = "../data/ct_match_corn_ALL_subsectors_2021_2024.csv"

emissions_county_tidy = tidy_emissions("../data/ct_match_corn_ALL_subsectors_2021_2024.csv", "../data/climateTRACE_tidy.csv")
emissions_county_tidy.head(5)

,year,state,admin,crop_fire_co2e_100yr,crop_fire_n2o,crop_residues_co2e_100yr,crop_residues_n2o,manure_fertilizer_co2e_100yr,manure_fertilizer_n2o,rice_co2e_100yr,rice_n2o,soil_co2e_100yr,soil_n2o,synth_fertilizer_co2e_100yr,synth_fertilizer_n2o
0,2021,ALABAMA,BALDWIN,8564.807217,6.516000e+00,4668.08706,17.09922,10456.16208,38.30096,575.899059,0.0,59019.547184,29.149181,45922.77690,168.21530
1,2021,ALABAMA,BARBOUR,589.283512,4.483202e-01,1090.00710,3.99270,4432.23144,16.23528,525.767233,0.0,33297.901576,15.278790,28706.69802,105.15274
2,2021,ALABAMA,BLOUNT,296.569885,2.256201e-01,4114.03356,15.06972,5180.99400,18.97800,552.459383,0.0,24484.589499,11.234789,29192.38686,106.93182
3,2021,ALABAMA,BUTLER,0.000046,1.491834e-07,308.93226,1.13162,1461.25434,5.35258,191.922970,0.0,29102.172163,14.373280,10626.19740,38.92380
4,2021,ALABAMA,CALHOUN,0.022775,2.011099e-05,698.57970,2.55890,1804.48632,6.60984,298.804317,0.0,21651.332000,10.693382,11389.65828,41.72036


# Scratch Work Below

In [ ]:
raw_df = pd.read_csv("../data/corn_state.csv")
raw_df.head(2)

In [ ]:
df_no_money = raw_df.loc[raw_df['unit'] != '$']

In [ ]:
df_usefulcols = df_no_money[["year", "state", "state_abbr", "commodity", "util_practice_desc", "statistic_category", "value"]]
df_usefulcols.head(2)

In [ ]:
df_pivot = df_usefulcols.pivot_table(index=["year", "state", "state_abbr", "util_practice_desc", "commodity"], columns="statistic_category", values="value", aggfunc="first")
df_pivot.columns.name=None
df_pivot = df_pivot.reset_index()
df_pivot.head(5)

In [ ]:
############ Doesn't work

# Split into the three groups
df_planted = df_usefulcols[df_usefulcols["util_practice_desc"] == "ALL UTILIZATION PRACTICES"]
df_harvest = df_usefulcols[df_usefulcols["util_practice_desc"].isin(["GRAIN", "SILAGE"])]

# Pivot each separately
planted_pivot = df_planted.pivot_table(index=["year", "state", "state_abbr", "commodity"], columns="statistic_category", values="value", aggfunc="first").reset_index()
planted_pivot.columns.name = None

# Will only have 'area planted', rename to be explicit
#planted_pivot = planted_pivot.rename(columns={"area planted": "area planted"})

harvest_pivot = df_harvest.pivot_table(
    index=["year", "state", "state_abbr", "commodity", "util_practice_desc"],
    columns="statistic_category",
    values="value",
    aggfunc="first"
).reset_index()
harvest_pivot.columns.name = None

# Merge area planted onto each grain/silage row
df_final = harvest_pivot.merge(
    planted_pivot[["year", "state", "state_abbr", "commodity", "AREA PLANTED"]],
    on=["year", "state", "state_abbr", "commodity"],
    how="left"
)
df_final.head(10)

In [ ]:
# Split into planted and harvest groups as before
df_planted = df_usefulcols[df_usefulcols["util_practice_desc"] == "ALL UTILIZATION PRACTICES"]
df_harvest = df_usefulcols[df_usefulcols["util_practice_desc"].isin(["GRAIN", "SILAGE"])]

# For harvest rows, combine the stat category and util practice into one label
df_harvest = df_harvest.copy()
df_harvest["stat_label"] = (
    df_harvest["statistic_category"] + " " + df_harvest["util_practice_desc"].str.lower()
)
# e.g. "area harvested grain", "production silage", "yield grain" etc.

df_harvest = df_harvest[df_harvest["statistic_category"] != "AREA PLANTED"]

# For planted rows, the label is just the stat category + total
df_planted = df_planted.copy()
df_planted["stat_label"] = df_planted["statistic_category"] + " total"
# e.g. "area planted total"

# Combine and pivot once on the new label
df_combined = pd.concat([df_harvest, df_planted], ignore_index=True)

df_final = df_combined.pivot_table(
    index=["year", "state"],
    columns="stat_label",
    values="value",
    aggfunc="first"
).reset_index()
df_final.columns.name = None

In [ ]:
df_final.head(10)

In [ ]:
len(df_usefulcols) / len(df_pivot)

In [ ]:
index_cols = ["year", "state", "statistic_category"]

duplicates = raw_df[raw_df.duplicated(subset=index_cols, keep=False)]
#print(duplicates.sort_values(index_cols))

In [ ]:
print(duplicates[["description", "class", "statistic_category"]].drop_duplicates())

### Tidy climate trace data

In [14]:
read_path = "../data/Copy of ct_match_corn_IOWA_subsectors_2021_2024.csv"
df = pd.read_csv(read_path)
df.head(10)

,sector,subsector,gas,emissionsQuantity,percentage,year,month,date,gadmId,admin,state
0,power,electricity-generation,co2e_100yr,994560.000000,78.474239,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
1,mineral-extraction,iron-mining,co2e_100yr,0.354633,0.000028,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
2,fossil-fuel-operations,oil-and-gas-production,co2e_100yr,0.000000,0.000000,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
3,agriculture,other-agricultural-soil-emissions,co2e_100yr,11911.653033,0.939871,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
4,power,heat-plants,co2e_100yr,0.000000,0.000000,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
5,buildings,other-onsite-fuel-usage,co2e_100yr,74.122459,0.005849,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
6,agriculture,enteric-fermentation-cattle-operation,co2e_100yr,10856.625377,0.856625,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
7,manufacturing,iron-and-steel,co2e_100yr,0.000000,0.000000,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
8,manufacturing,pulp-and-paper,co2e_100yr,119.515012,0.009430,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
9,forestry-and-land-use,forest-land-fires,co2e_100yr,6017.589000,0.474809,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA


In [27]:
df_agr = df[df["sector"] == "agriculture"].copy()
df_agr.head(10)

,sector,subsector,gas,emissionsQuantity,percentage,year,month,date,gadmId,admin,state
3,agriculture,other-agricultural-soil-emissions,co2e_100yr,11911.653033,0.939871,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
6,agriculture,enteric-fermentation-cattle-operation,co2e_100yr,10856.625377,0.856625,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
11,agriculture,manure-applied-to-soils,co2e_100yr,3669.286530,0.289519,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
32,agriculture,cropland-fires,co2e_100yr,56.025169,0.004421,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
37,agriculture,manure-left-on-pasture-cattle,co2e_100yr,2039.454480,0.160920,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
38,agriculture,manure-management-other,co2e_100yr,5075.890856,0.400505,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
45,agriculture,synthetic-fertilizer-application,co2e_100yr,23332.551060,1.841019,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
46,agriculture,crop-residues,co2e_100yr,6935.985420,0.547273,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
47,agriculture,enteric-fermentation-other,co2e_100yr,1730.659119,0.136555,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA
48,agriculture,manure-management-cattle-operation,co2e_100yr,11015.422326,0.869155,2021,NaN,NaN,USA.16.3_1,ALLAMAKEE,IOWA


In [20]:
len(df)/len(df_agr)

5.333333333333333

In [21]:
print(len(set(df_agr["subsector"].tolist())))
print(list(set(df_agr["subsector"].tolist())))

12
['crop-residues', 'enteric-fermentation-cattle-pasture', 'manure-applied-to-soils', 'enteric-fermentation-cattle-operation', 'synthetic-fertilizer-application', 'enteric-fermentation-other', 'rice-cultivation', 'manure-left-on-pasture-cattle', 'manure-management-cattle-operation', 'other-agricultural-soil-emissions', 'cropland-fires', 'manure-management-other']


In [28]:
subsector_names = {
    "crop-residues": "crop_residues",
    "enteric-fermentation-cattle-pasture": "livestock",
    "enteric-fermentation-cattle-operation": "livestock",
    "enteric-fermentation-other": "livestock",
    "manure-left-on-pasture-cattle": "livestock",
    "manure-applied-to-soils": "manure_fertilizer",
    "manure-management-cattle-operation": "livestock",
    "manure-management-other": "livestock",
    "synthetic-fertilizer-application": "synth_fertilizer",
    "other-agricultural-soil-emissions": "soil",
    "rice-cultivation": "rice",
    "cropland-fires": "crop_fire"
}

df_agr["source"] = [subsector_names[subsector] for subsector in df_agr["subsector"]]
df_agr = df_agr[df_agr["source"] != "livestock"][["year", "state", "admin", "source", "gas", "emissionsQuantity"]]
df_agr.head(10)

,year,state,admin,source,gas,emissionsQuantity
3,2021,IOWA,ALLAMAKEE,soil,co2e_100yr,11911.653033
11,2021,IOWA,ALLAMAKEE,manure_fertilizer,co2e_100yr,3669.286530
32,2021,IOWA,ALLAMAKEE,crop_fire,co2e_100yr,56.025169
45,2021,IOWA,ALLAMAKEE,synth_fertilizer,co2e_100yr,23332.551060
46,2021,IOWA,ALLAMAKEE,crop_residues,co2e_100yr,6935.985420
50,2021,IOWA,ALLAMAKEE,rice,co2e_100yr,144.234046
71,2021,IOWA,ALLAMAKEE,crop_fire,co2e_100yr,0.000000
72,2021,IOWA,ALLAMAKEE,crop_residues,co2e_100yr,577.799040
94,2021,IOWA,ALLAMAKEE,manure_fertilizer,co2e_100yr,305.667180
102,2021,IOWA,ALLAMAKEE,soil,co2e_100yr,845.067013


In [31]:
df_agr = df_agr.copy()
df_agr["emission"] = (
    df_agr["source"] + "_" + df_agr["gas"].str.lower()
)
df_agr.head(10)

,year,state,admin,source,gas,emissionsQuantity,emission
3,2021,IOWA,ALLAMAKEE,soil,co2e_100yr,11911.653033,soil_co2e_100yr
11,2021,IOWA,ALLAMAKEE,manure_fertilizer,co2e_100yr,3669.286530,manure_fertilizer_co2e_100yr
32,2021,IOWA,ALLAMAKEE,crop_fire,co2e_100yr,56.025169,crop_fire_co2e_100yr
45,2021,IOWA,ALLAMAKEE,synth_fertilizer,co2e_100yr,23332.551060,synth_fertilizer_co2e_100yr
46,2021,IOWA,ALLAMAKEE,crop_residues,co2e_100yr,6935.985420,crop_residues_co2e_100yr
50,2021,IOWA,ALLAMAKEE,rice,co2e_100yr,144.234046,rice_co2e_100yr
71,2021,IOWA,ALLAMAKEE,crop_fire,co2e_100yr,0.000000,crop_fire_co2e_100yr
72,2021,IOWA,ALLAMAKEE,crop_residues,co2e_100yr,577.799040,crop_residues_co2e_100yr
94,2021,IOWA,ALLAMAKEE,manure_fertilizer,co2e_100yr,305.667180,manure_fertilizer_co2e_100yr
102,2021,IOWA,ALLAMAKEE,soil,co2e_100yr,845.067013,soil_co2e_100yr


In [34]:
df_pivot = df_agr.pivot_table(
    index=["year", "state", "admin"],
    columns="emission",
    values="emissionsQuantity",
    aggfunc="sum"
).reset_index()
df_pivot.columns.name = None

In [35]:
df_pivot.head(10)

,year,state,admin,crop_fire_co2e_100yr,crop_fire_n2o,crop_residues_co2e_100yr,crop_residues_n2o,manure_fertilizer_co2e_100yr,manure_fertilizer_n2o,rice_co2e_100yr,rice_n2o,soil_co2e_100yr,soil_n2o,synth_fertilizer_co2e_100yr,synth_fertilizer_n2o
0,2021,IOWA,ADAIR,686.107752,0.52198,24978.24966,91.49542,9678.10662,35.45094,222.609259,0.0,21406.201062,9.822266,59292.44412,217.18844
1,2021,IOWA,ALLAMAKEE,112.050337,0.08524,13871.97084,50.81308,7338.57306,26.88122,288.468092,0.0,23823.306067,10.931359,46665.10212,170.93444
2,2021,IOWA,APPANOOSE,1187.220739,0.90322,14821.46484,54.29108,6458.35008,23.65696,249.381695,0.0,18943.490323,8.692248,35968.96758,131.75446
3,2021,IOWA,AUDUBON,162.533543,0.12366,24829.71036,90.95132,9135.90132,33.46484,236.427257,0.0,16623.297548,8.210085,61273.54506,224.44522
4,2021,IOWA,BENTON,619.207293,0.47108,40943.78106,149.97722,15912.94614,58.28918,315.776116,0.0,26833.285872,12.312492,100586.54424,368.44888
5,2021,IOWA,BLACK HAWK,286.377929,0.21788,32277.71274,118.23338,12888.94152,47.21224,313.705103,0.0,19895.510451,9.826199,82866.64932,303.54084
6,2021,IOWA,BOONE,175.110271,0.13324,32560.45338,119.26906,12477.85266,45.70642,299.531875,0.0,21337.792443,10.538528,82658.43768,302.77816
7,2021,IOWA,BREMER,182.197163,0.13862,23628.13908,86.54996,9885.40644,36.21028,237.243605,0.0,16201.444321,7.434056,62734.73388,229.79756
8,2021,IOWA,BUCHANAN,369.344973,0.28100,33798.53022,123.80414,14456.08164,52.95268,237.957828,0.0,21337.792444,9.790877,91768.74798,336.14926
9,2021,IOWA,BUENA VISTA,154.259133,0.11736,30642.76488,112.24456,10698.23664,39.18768,286.005606,0.0,21548.719045,9.887661,81949.76244,300.18228
